# Data Inspection

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
import tabulate
from IPython.display import display, Markdown
from pathlib import Path

# Custom packages
from tools.filter import FilterDF as fdf
from tools.benchmarks import ParetoAnalysis as pa
from tools.benchmarks import AccuracyCalculation as ac
from tools.integrity_fixes import DataFixer as fix, DataExporter as exporter
from tools.coverage_functions import coverage_calculator, plot_time_series, plot_time_spacing

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Read data from parquet files

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()

# Data already exists
else:
    static_data = static_data_merged.copy()
    
%store static_data_merged

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()
        
%store sales_data_merged

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
%store -r before_after_details_true
if 'before_after_details_true' not in locals():
    before_after_details_true = pd.read_csv('data/before_after_details_true.csv', index_col='location_id')
%store before_after_details_true

# Timezones
%store -r timezones
if 'timezones' not in locals():
    timezones = pd.read_csv('data/timezones.csv', index_col='location_id')['timezone'].to_dict()
    for loc_id, df in sales_and_menu_data.items():
        df.index = df.index.tz_convert(timezones[loc_id])
        sales_and_menu_data[loc_id] = df
%store timezones

Timezones

### Data Density: Entire Set, Before Promo, and After Promo

In [ ]:
# Initialize a list to see if there is a data buffer before and after the promotional item introduction
data_coverage_list_4m = []
data_coverage_list_b2m = []
data_coverage_list_a2m = []
data_coverage_list_all = []
data_coverage_list_before = []
data_coverage_list_after = []

for loc_id, df in sales_and_menu_data.items():
    row_4m = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['4mo'])
    row_b2m = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['b2m'])
    row_a2m = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['a2m'])
    row_all = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['all'])
    row_before = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['bef'])
    row_after = coverage_calculator(loc_id, df, before_after_details, timezones, periods=['aft'])
    
    data_coverage_list_4m.append(row_4m)
    data_coverage_list_b2m.append(row_b2m)
    data_coverage_list_a2m.append(row_a2m)
    data_coverage_list_all.append(row_all)
    data_coverage_list_before.append(row_before)
    data_coverage_list_after.append(row_after)

# Create data frame
data_coverage_4m = pd.DataFrame(data_coverage_list_4m).sort_values('4mo_12H_cover', ascending=False).set_index('loc_id')
restaurants_by_4m_coverage = data_coverage_4m.index.tolist()
data_coverage_b2m = pd.DataFrame(data_coverage_list_b2m).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_a2m = pd.DataFrame(data_coverage_list_a2m).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_all = pd.DataFrame(data_coverage_list_all).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_before = pd.DataFrame(data_coverage_list_before).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_after = pd.DataFrame(data_coverage_list_after).set_index('loc_id').loc[restaurants_by_4m_coverage]

# Reorder
before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'] = before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'].str.title()

View

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)
data_coverage_all.loc[restaurants_by_4m_coverage]

In [ ]:
%store -r time_differences
%store -r time_differences_details

if 'time_differences_details' not in locals() or 'time_differences' not in locals():

    time_differences_details = {}
    time_differences = {}
    for loc_id in restaurants_by_4m_coverage:

        df = sales_and_menu_data[loc_id]

        # Group by transactions (at the same time)
        transactions = df.groupby('created_at').agg({'item_quantity' : 'sum'})

        # Group by individuals days and days of the week
        transaction_by_dayofweek = transactions.groupby([transactions.index.dayofweek, transactions.index.date])

        # Take the index at every group and find the difference between time points (dropping the NaT edges) and convert to hours
        time_diffs_on_dayofweek = transaction_by_dayofweek.apply(lambda s: s.index.to_series().diff().dropna().dt.seconds//3600)

        existing_combinations = time_diffs_on_dayofweek.index.drop_duplicates()

        # Create a new MultiIndex from all days and existing (date, datetime) combinations
        all_days = np.arange(7) 
        new_indices = [(day, date, datetime) for day in all_days for _, date, datetime in existing_combinations]
        time_diffs_on_dayofweek = time_diffs_on_dayofweek.reindex(new_indices)

        # Store entire pivoted data
        time_differences_details[loc_id] = time_diffs_on_dayofweek

        time_diff_frequencies_list = []
        for dayofweek in range(7):

            # Subset to given day of the week and calculate the frequencies
            if dayofweek in time_diffs_on_dayofweek.index.get_level_values(0):
                time_diff_frequencies_specific_day = time_diffs_on_dayofweek[dayofweek].value_counts()
                time_diff_frequencies_specific_day.index.name = "time_diffs"
                time_diff_frequencies_specific_day.name = dayofweek
                time_diff_frequencies_list.append(pd.DataFrame(time_diff_frequencies_specific_day))

        time_diff_frequencies = time_diff_frequencies_list[0].join(time_diff_frequencies_list[1:], how='outer').sort_index()
        time_diff_frequencies = time_diff_frequencies.rename(columns={0:'Monday', 1:'Tuesday', 2:'Wednesday', 3:'Thursday', 4:'Friday', 5:'Saturday', 6:'Sunday'})
        time_diff_frequencies.columns.name = loc_id

        time_differences[loc_id] = time_diff_frequencies

import pickle
# export as pickle
with open('data/2_palate_data_parquet_cleaned/time_differences.pkl', 'wb') as f:
    pickle.dump(time_differences, f)
    
with open('data/2_palate_data_parquet_cleaned/time_differences_details.pkl', 'wb') as f:
    pickle.dump(time_differences_details, f)

%store time_differences
%store time_differences_details

In [ ]:
plt.plot(sales_and_menu_data['L69HYJ4Y3TR91'][sales_and_menu_data['L69HYJ4Y3TR91']['item_modifications'].str.lower().str.contains('impossible')].resample('W')['item_quantity'].sum())

In [ ]:
plt.plot(sales_and_menu_data['ED5J990H5VAZT'].dropna(subset='item_modifications')[sales_and_menu_data['ED5J990H5VAZT']['item_modifications'].dropna().str.lower().str.contains('vegan') & sales_and_menu_data['ED5J990H5VAZT']['item_modifications'].dropna().str.lower().str.contains('bacon')].resample('W')['item_quantity'].sum())

Bar

In [ ]:
sales_and_menu_data['EMBVNVD207CC6'][(sales_and_menu_data['EMBVNVD207CC6']['item_type'] != 'Drink') & (sales_and_menu_data['EMBVNVD207CC6']['is_plant_based'] != 'Unsure')]['item_name'].value_counts()

In [ ]:
fdf(sales_and_menu_data['EMBVNVD207CC6']).substring('item_modifications', 'Vegan')


Queso

In [ ]:
fdf(sales_and_menu_data['AQD04SM0J92WA']).substring('item_modifications', 'Vegan')
# fdf(sales_and_menu_data['AQD04SM0J92WA']).substring('item_name', 'Vegan')

In [ ]:
plt.plot(sales_and_menu_data['75WYSXR9QBK5M'][sales_and_menu_data['75WYSXR9QBK5M']['item_name'].str.lower().str.contains('kimchee')].resample('W')['item_quantity'].sum())

In [ ]:
plt.plot(sales_and_menu_data['V3Q26BHF3SE2H'][sales_and_menu_data['V3Q26BHF3SE2H']['item_modifications'].str.lower().str.contains('beyond')].resample('W')['item_quantity'].sum())

In [ ]:
plt.plot(sales_and_menu_data['LBZEEFSBJNB3Z'][sales_and_menu_data['LBZEEFSBJNB3Z']['item_modifications'].str.lower().str.contains('vegan') & sales_and_menu_data['LBZEEFSBJNB3Z']['item_name'].str.lower().str.contains('wake')].resample('W')['item_quantity'].sum())

In [ ]:
sales_and_menu_data['LBMCPAYT7W36V'][sales_and_menu_data['LBMCPAYT7W36V']['item_name'].str.lower().str.contains('vegan') | sales_and_menu_data['LBMCPAYT7W36V']['item_modifications'].str.lower().str.contains('vegan')]

In [ ]:
plt.plot(fdf(sales_and_menu_data['S8MT0YGD2KTN9']).substring('item_name','vurger').resample('W')['item_quantity'].sum())

In [ ]:
pd.concat([fdf(sales_and_menu_data['1SQPTEGYPH0GA']).substring('item_modifications','impossible'), fdf(sales_and_menu_data['1SQPTEGYPH0GA']).substring('item_name','impossible')])

In [ ]:
plt.plot(fdf(sales_and_menu_data['9XKJD8DQTH559']).substring('item_modifications','impossible').resample('W')['item_quantity'].sum())

In [ ]:
before_after_details.loc[restaurants_by_4m_coverage]

In [ ]:
sales_and_menu_data['JHDN7CF1C03X5']

In [ ]:
restaurants_by_4m_coverage

In [ ]:
%store -r items_tagged_new
top_80_percent = items_tagged_new

In [ ]:
pba_substring_set = ['Beyond', 'Impossible', 'Vegan', 'Veggie', 'Jackfruit']
def filter_substrings(text):
    matches = [substr for substr in pba_substring_set if substr in text]
    return ' '.join(matches)
pba_substrings = before_after_details['first_plant_based_mention'].apply(filter_substrings).rename('substring')
before_after_details_substrings = pd.concat([before_after_details, pba_substrings], axis=1)

In [ ]:
item_in_top_total = 0
top_items_from_sales = []
top_item_percentages = []

for loc_id in tqdm(restaurants_by_4m_coverage):

    # Setup
    top_items = set(top_80_percent.query('location_id in @loc_id')['item_name'].tolist())
    df = sales_and_menu_data[loc_id]
    promo_word = before_after_details.loc[loc_id,'first_plant_based_mention']
    
    # Naive
    item_in_top_total += int(promo_word in top_items)
    
    
    ######
    
    top_item_sales = df[df['item_name'].isin(top_items)]
    
    top_modifications = top_item_sales['item_modifications']
    top_items = top_item_sales['item_name']
    
    promo_substring = before_after_details_substrings.loc[loc_id, 'substring']
    
    df_containing_promo = df[df['item_name'].str.contains(promo_substring) | df['item_modifications'].str.contains(promo_substring)]
    
    potential_pbas = df_containing_promo['item_name'].drop_duplicates()
    
    top_80_item_percent = potential_pbas.isin(top_items).sum() / potential_pbas.size
    
    top_80_modification_percent = potential_pbas.isin(top_modifications).sum() / potential_pbas.size
    
    top_item_percentages.append((loc_id, top_80_item_percent, top_80_modification_percent))
    
    #top_items_from_sales.append(top_item_sales)
    #top_items_from_sales += df[df['item_name'].isin(top_items)]['item_name'].unique().tolist()
    
pd.DataFrame(top_item_percentages, columns=['location_id', 'pct_in_top_items', 'pct_in_top_modifications'])

In [ ]:
before_after_details.loc[restaurants_by_4m_coverage]

In [ ]:
sales_and_menu_data[loc_id].query('item_name == "Jackfruits"', engine='python')['item_modifications'].unique()

In [ ]:
true_promos = pd.DataFrame(zip(location_ids, ['Impossible',
                   'Beyond Sausage',
                   'Beyond Burger',
                   'Impossible Sausage',
                   ['Vegan','Bacon'], # and
                   'Vegan Breakfast Sandwich',
                   'Vegan',
                   ['Vegan','Queso'], # and
                   'Beyond',
                   'Kimchee Veggie Burger',
                   'Beyond Burger',
                   'Wake and Fake',
                   ['Vegan Sausage','Impossible Sausage'], # or
                   'Chille Verde Jackfruit', # (as name not modification)
                   'Beyond Meat Patty Melt',
                   'Vegan Elvis',
                   'Vegan',
                   'Better Than Beyond',
                   'Impossible',
                   'Impossible',
                   'Beyond Burger',
                   'Veggie Sausage',
                   'Beyond',
                   'Vegan',
                   'Beyond', # N/A
                   'Arkie Vegan',
                   'Impossible Orbit',
                   'Vegan Burger',
                   'Impossible Burger',
                   'Impossible Burger']))
true_promos

In [ ]:
k=22
for loc_id in restaurants_by_4m_coverage[k:k+1]:
    
    promo = before_after_details.loc[loc_id, 'first_plant_based_mention']
    promo_substring = before_after_details_substrings.loc[loc_id, 'substring']
    
    df = sales_and_menu_data[loc_id]
    df_containing_promo = df[df['item_name'].str.contains(promo_substring) | df['item_modifications'].str.contains(promo_substring)]
    #potential_promo_sales = df_containing_promo.drop_duplicates(subset=['item_name','item_modifications'])
    potential_promos = df_containing_promo.value_counts(subset=['item_name','item_modifications'])
    
    print(loc_id, promo)
    print(potential_promos.to_string())
    print("\n\n\n\n")

In [ ]:
locations.loc[restaurants_by_4m_coverage].loc[[loc_id]]

In [ ]:
print(sales_and_menu_data[loc_id]['item_name'].value_counts().to_string())

In [ ]:
before_after_details.loc['ED5J990H5VAZT']

In [ ]:
sales_and_menu_data['ED5J990H5VAZT'].query('(item_name.str.contains("Bacon") | item_modifications.str.contains("Bacon")) & is_plant_based == "Yes"', engine='python')[['item_name','item_modifications']].value_counts()

In [ ]:
sales_and_menu_data['ED5J990H5VAZT'].query('item_name.str.contains("Bacon") & item_name.str.contains("Vegan") | item_modifications.str.contains("Bacon") & item_modifications.str.contains("Vegan")', engine='python')[['item_name','item_modifications']]

In [ ]:
sales_and_menu_data['ED5J990H5VAZT'].query('item_name.str.contains("Thrilling")', engine='python')

In [ ]:
sales_and_menu_data['ED5J990H5VAZT'].query('brand == "Black Sheep" | brand == "Thrilling"')

In [ ]:
print(sales_and_menu_data['ED5J990H5VAZT']['item_name'].value_counts().to_string())

In [ ]:
sales_and_menu_data['ED5J990H5VAZT'].loc[before_after_details.loc['ED5J990H5VAZT','cross_over_date']:].query("(item_name.str.contains('Vegan') or item_modifications.str.contains('Vegan')) and (item_modifications.str.contains('Bacon') or item_name.str.contains('Alchemist') or item_name.str.contains('Yeti') or item_name.str.contains('Yeli') or item_name.str.contains('Anam A'))", engine='python')

In [ ]:
sales_and_menu_data['ED5J990H5VAZT'].query('item_name.str.contains("Yeli")', engine='python')

In [ ]:
print(sales_and_menu_data['ED5J990H5VAZT'][['item_name','item_modifications']].value_counts().to_string())

In [ ]:
sales_and_menu_data['L69HYJ4Y3TR91'].query('(item_name.str.contains("Vegan") | item_modifications.str.contains("Vegan")) & ~item_modifications.str.contains("Vegan Rosemary Bagel")')[['item_name','item_modifications']].value_counts()

In [ ]:
sales_and_menu_data['L69HYJ4Y3TR91'].query('(item_name.str.contains("Impossible") | item_modifications.str.contains("Impossible"))')[['item_name','item_modifications']].value_counts()#.resample('W')['item_quantity'].sum().plot()

In [ ]:
sales_and_menu_data['1SQPTEGYPH0GA'].resample('D')['item_quantity'].sum().plot()

In [ ]:
sales_and_menu_data['LBZEEFSBJNB3Z'].query('item_name.str.contains("Wake")')

In [ ]:
sales_and_menu_data['V3Q26BHF3SE2H'].query('item_name.str.contains("Beyond") & item_modifications.str.contains("Beyond")')

In [ ]:
print(sales_and_menu_data['JHDN7CF1C03X5'].query('item_name.str.contains("Beyond Burger") | item_modifications.str.contains("Beyond Burger")')[['item_name','item_modifications']].value_counts().to_string())

In [ ]:
before_after_details.loc[restaurants_by_4m_coverage]

In [ ]:
'V3Q26BHF3SE2H' # missing (1 sale) PBA
'MS8R16DY0JQAM' # missing PBA

In [ ]:
print(sales_and_menu_data['V3Q26BHF3SE2H'].query('item_name.str.contains("beyond") or item_modifications.str.contains("Beyond")', engine='python')[['item_name','item_modifications']].value_counts().to_string())

In [ ]:
sales_and_menu_data['MS8R16DY0JQAM'].query('item_name.str.lower().str.contains("beyond") or item_modifications.str.lower().str.contains("beyond") or item_name.str.lower().str.contains("vegan") or item_modifications.str.lower().str.contains("vegan")', engine='python')

In [ ]:
before_after_details_true

In [ ]:
print(sales_and_menu_data['S8MT0YGD2KTN9']['item_name'].value_counts().to_string())

In [ ]:
print(sales_and_menu_data['S8MT0YGD2KTN9']['item_modifications'].value_counts().to_string())

In [ ]:
promo_datetime = before_after_details_true.loc['S8MT0YGD2KTN9','cross_over_date']
promo_list = before_after_details_true.loc['S8MT0YGD2KTN9','promo_name']
promo_item_containing = sales_and_menu_data['S8MT0YGD2KTN9'][sales_and_menu_data['S8MT0YGD2KTN9']['item_name'].str.contains(promo_list) | sales_and_menu_data['S8MT0YGD2KTN9']['item_modifications'].str.title().str.contains(promo_list)]
plt.plot(promo_item_containing.resample('W')['item_quantity'].sum())
plt.axvline(x=promo_datetime, color='red', linestyle='--', label='Promo Date')
plt.title("Plant-Based Analog")
plt.xticks(rotation=70)
plt.show()

In [ ]:
vurger_conditions = ['item_modifications.str.title().str.contains("Vegan")', 
                     'item_modifications.str.title().str.contains("Vurger")',
                     'item_modifications.str.title().str.contains("V-Urger")',
                     'item_name.str.title().str.contains("Vurger")',
                     'item_name.str.title().str.contains("V-Urger")']
print(sales_and_menu_data['S8MT0YGD2KTN9'].query(' or '.join(vurger_conditions))[['item_name','item_modifications']].to_string())

In [ ]:
plot_time_series('S8MT0YGD2KTN9', 
                 sales_and_menu_data['S8MT0YGD2KTN9'], 
                 (before_after_details_true
                  .assign(cross_over_date = lambda df: df['cross_over_date']
                          .mask(df.index == 'S8MT0YGD2KTN9', pd.Timestamp('2020-07-01 00:00:00-04:00')))), 
                 max_ylim=1000)

In [ ]:
sales_and_menu_data['SRQS8F7JWA9MZ'].query('item_name.str.lower().str.contains("impossible") or item_modifications.str.lower().str.contains("impossible")')

In [ ]:
sales_and_menu_data['SRQS8F7JWA9MZ'].query('item_name.str.lower().str.contains("beyond") or item_modifications.str.lower().str.contains("beyond")')

In [ ]:
sales_and_menu_data['CB2KHY1C2G9PT'].query('item_name.str.lower().str.contains("impossible") or item_modifications.str.lower().str.contains("impossible")')

In [ ]:
sales_and_menu_data['CB2KHY1C2G9PT'].query('item_name.str.lower().str.contains("beyond") or item_modifications.str.lower().str.contains("beyond")')

In [ ]:
sales_and_menu_data['SRQS8F7JWA9MZ']['item_name'].value_counts()

In [ ]:
print(sales_and_menu_data['CB2KHY1C2G9PT']['item_name'].value_counts().to_string())

In [ ]:
sales_and_menu_data['2HRX9P6HKXA8V'].query('item_name.str.lower().str.contains("beyond") or item_modifications.str.lower().str.contains("beyond")')['item_quantity'].resample('W').sum().plot()

In [ ]:
df['item_name'].value_counts(ascending=True)

In [ ]:
# Set up df
loc_id = 'SRQS8F7JWA9MZ'
df = sales_and_menu_data[loc_id].assign(item_name = lambda df: df['item_name'].str.title())

# Visualizing with gaps for inactive weeks
introduction_fig, ax = plt.subplots(figsize=(14, 8))

# Index into the promotional items for this restaurant
promo_datetime = before_after_details.loc[loc_id,'cross_over_date'].tz_convert('UTC')

for dish in df['item_name'].value_counts(ascending=False).index[:50][::-1]:
    dish_df = df.query('item_name.str.contains(@dish)')
    dish_activity = (dish_df['item_quantity']
                                    .resample('W')
                                    .sum()
                                    .to_frame(name='W')
                                    .query('0 < W')
                                    .index
                                    .tz_localize(None)
                                    .to_period('W')
                                    .tolist())

    # For every active week
    for week in dish_activity:

        # Place a blue dot
        ax.hlines(y=dish, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=loc_id)

# Place a red circle for the promotional item
ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title('Introduction Weekly Activity for Each Dish')
ax.set_xlabel('Date')
ax.set_ylabel('Dish')

# Figure
introduction_fig.tight_layout()

plt.show()

In [ ]:
sales_and_menu_data['SRQS8F7JWA9MZ'].query('item_name.str.lower().str.contains("alternative")')